# SpaCal-Depth: Metric3D v2 and ETH3D caching on Colab

Runtime → Change runtime type → **A100** (L4 / T4: set `AMP = True`).

Before the first session: upload a fresh `spacal_code.tgz` to `MyDrive/spacal/` (it must contain `src/models/metric3d.py` and `src/data/eth3d.py`).

Jobs: Metric3D v2 (ViT-Large) on NYU, KITTI, DIODE indoor/outdoor and ETH3D; UniDepthV2 and Depth Pro on ETH3D. Everything goes to separate Drive folders (`cache_new`, `prepared_new`, `results_new`), so earlier caches and results are never touched. Resumable: after a disconnect, Run all again.

ETH3D is streamed (49 GB download, nothing large on disk) and kept 4× reduced, ~4 GB, in `MyDrive/spacal/data/eth3d`, so it survives disconnects; the stream resumes where it stopped.

Output to bring back: `MyDrive/spacal/prepared_new/*.npz` (→ cluster `prepared/`) and `MyDrive/spacal/results_new/e0/per_image.csv`.

In [ ]:
# 1. GPU + Drive
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc && free -g | head -2 && df -h /content | tail -1
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Configuration: (dataset, model job) pairs; trim to resume selectively
import os
JOBS = [
    ('nyu', 'metric3d-large --flip'),
    ('kitti', 'metric3d-large --flip'),
    ('diode_indoor', 'metric3d-large --flip'),
    ('diode_outdoor', 'metric3d-large --flip'),
    ('eth3d', 'metric3d-large --flip'),
    ('eth3d', 'unidepthv2-large --flip'),
    ('eth3d', 'depthpro --flip'),
]
AMP = False              # A100: full precision. L4 / T4: True
PRUNE_AFTER = False      # True deletes raw caches once their tables exist (keeps RGB samples)

DRIVE = '/content/drive/MyDrive/spacal'
os.makedirs(f'{DRIVE}/logs', exist_ok=True)
os.environ.update({
    'SPACAL_DATA_ROOT': '/content/data',
    'SPACAL_CACHE_ROOT': f'{DRIVE}/cache_new',
    'SPACAL_PREPARED_ROOT': f'{DRIVE}/prepared_new',
    'SPACAL_RESULTS_ROOT': f'{DRIVE}/results_new',
    'HF_HOME': '/content/hf_cache',
    'TORCH_HOME': '/content/torch_cache',
    'SPACAL_DEVICE': 'cuda',
    'PYTHONUNBUFFERED': '1',
})
DATASETS = sorted({ds for ds, _ in JOBS})
TAGS = lambda ds: [job.split()[0] for d, job in JOBS if d == ds]
DL_TARGETS = sorted({'nyu': 'nyu', 'kitti': 'kitti', 'diode_indoor': 'diode', 'diode_outdoor': 'diode',
                     'eth3d': 'eth3d'}[d] for d in DATASETS if d != 'eth3d')
AMP_ENV = 'AMP=1 ' if AMP else ''
print('jobs:', JOBS, '\ndownloads:', DL_TARGETS)

In [ ]:
# 3. Code
!rm -rf /content/CV_Project && tar xzf {DRIVE}/spacal_code.tgz -C /content
%cd /content/CV_Project
!test -f src/models/metric3d.py -a -f src/data/eth3d.py && echo 'code ok' || echo '!!! old tarball: upload a fresh spacal_code.tgz'

In [ ]:
# 4. Dependencies (Colab's torch is kept). Metric3D needs mmengine (mmcv is stubbed by our wrapper).
!pip install -q -r requirements-colab.txt mmengine
!pip install -q --no-deps git+https://github.com/lpiccinelli-eth/UniDepth.git
!python -c "import torch, transformers, unidepth, mmengine; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| transformers', transformers.__version__)"
# downloads the Metric3D repo and weights into TORCH_HOME and proves the model builds
!python -m scripts.prefetch_models --only-metric3d

In [ ]:
# 5. Data. NYU/KITTI/DIODE on the local disk (~8 GB, ~15-30 min); ETH3D on Drive (streams 49 GB, keeps ~4 GB;
# resumes after a disconnect). Each target fails on its own; rerun the cell to resume a failed one.
if DL_TARGETS:
    !bash scripts/download_data.sh {' '.join(DL_TARGETS)}
if 'eth3d' in DATASETS:
    !mkdir -p {DRIVE}/data/eth3d /content/data && ln -sfn {DRIVE}/data/eth3d /content/data/eth3d
    !bash scripts/download_data.sh eth3d

In [ ]:
# 6. Smoke test (3 images per job) + scale sanity check.
# median(pred / gt) should be roughly 0.6-1.6; a value off by ~10x means the focal length or the
# canonical-camera conversion is wrong: stop and report it.
for ds, job in JOBS:
    !MODEL_JOBS="{job}" {AMP_ENV} LIMIT=3 bash scripts/cache_dataset.sh {ds}
import numpy as np
from src.data.cache import list_samples, load_gt, load_pred
for ds in DATASETS:
    for tag in TAGS(ds):
        try:
            sids = list_samples(ds, tag)[:3]
        except FileNotFoundError:
            print(f'{ds:14s} {tag:17s} !!! no cached samples (see the errors above)'); continue
        for sid in sids:
            g, p = load_gt(ds, sid), load_pred(ds, tag, sid)
            m = g['mask'].astype(bool)
            print(f"{ds:14s} {tag:17s} {sid[:36]:36s} median pred/gt = {np.median(p['pred'][m] / g['gt'][m]):.3f}"
                  f"  valid px = {m.mean():.2f}  t_pred = {float(p['t_pred']):.2f}s")

In [ ]:
# 7. Full caching (hours; keep the tab open). Safe to re-run after a disconnect.
for ds, job in JOBS:
    tag = job.split()[0]
    !MODEL_JOBS="{job}" {AMP_ENV} bash scripts/cache_dataset.sh {ds} 2>&1 | tee -a {DRIVE}/logs/cache_new_{ds}__{tag}.log

In [ ]:
# 8. Status: a (dataset, model) is COMPLETE when its manifest finished a full (non-smoke) run
import json
complete = []
for ds, job in JOBS:
    tag = job.split()[0]
    man = f"{os.environ['SPACAL_CACHE_ROOT']}/{ds}/{tag}/manifest.json"
    if not os.path.exists(man):
        print(f'{ds:14s} {tag:17s} not started'); continue
    m = json.load(open(man))
    done = 'finished' in m and not m['args']['limit'] and m['n_cached'] >= m['n_samples']
    complete += [(ds, tag)] if done else []
    print(f"{ds:14s} {tag:17s} {m.get('n_cached', '?')}/{m['n_samples']}", 'COMPLETE' if done else '')
!du -sh {DRIVE}/cache_new 2>/dev/null

In [ ]:
# 9. E0 metrics + PixelTables for completed pairs (+ optional prune once a dataset is fully done)
if complete:
    e0_datasets = ' '.join(sorted({ds for ds, _ in complete}))  # IPython's ! cannot expand nested braces
    !python -m experiments.e0_base_metrics --datasets {e0_datasets}
    for ds, tag in complete:
        !python -m experiments.prepare_tables --dataset {ds} --model {tag}
    for ds in DATASETS:
        if PRUNE_AFTER and all((ds, t) in complete for t in TAGS(ds)):
            !python -m scripts.prune_cache --dataset {ds} --yes
else:
    print('nothing complete yet')
!ls -la {DRIVE}/prepared_new

## After all jobs are finished

**Disconnect the GPU runtime.** Copy `MyDrive/spacal/prepared_new/*.npz` into the cluster's `prepared/` and `results_new/e0/per_image.csv` to the laptop, then on the cluster:

`sbatch -p phd --export=ALL,PARTS="metric3d eth3d" slurm/new.sbatch`

(The table step sees the tables already exist and goes straight to the main study, E5 and E11.)